# KrimbaGram · Export RVC model to ONNX (for the Android app)

Converts your `lecturer_ru.pth` into the two ONNX files the phone needs:
**`generator.onnx`** (your voice) + **`hubert.onnx`** (shared ContentVec encoder, downloaded).

**Before running:** **Internet: On** · add the **`HF_TOKEN`** secret (read token; the model repo is private). Run top to bottom. The 3.10 env build (cells 3-4) takes a few minutes the first time.

In [ ]:
# 1) config
HF_REPO  = "cyttic/lecturer-ru-rvc"
PTH_FILE = "lecturer_ru.pth"
NAME     = "lecturer_ru"
RVC      = "/kaggle/working/rvc"
OUT      = "/kaggle/working/android_models"
import sys; print("kernel python", sys.version.split()[0])

In [ ]:
# 2) clone RVC-WebUI
%cd /kaggle/working
![ -d rvc ] || git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI rvc

In [ ]:
# 3) Python 3.10 env via uv (RVC needs 3.10; downloads standalone CPython, no conda)
!pip install -q uv
!uv venv --seed --python 3.10 /kaggle/working/rvc310
!/kaggle/working/rvc310/bin/python --version

In [ ]:
# 4) install deps into the 3.10 env (incl. onnx/onnxruntime for export)
PY = "/kaggle/working/rvc310/bin/python"
!{PY} -m pip install -q "pip<24.1"
!{PY} -m pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cpu
!{PY} -m pip install -q numpy==1.26.4 faiss-cpu praat-parselmouth pyworld torchcrepe ffmpeg-python av tensorboardX fairseq librosa soundfile scipy onnx onnxruntime onnxsim onnxscript
!{PY} -c "import fairseq, onnx, onnxruntime, torch; print('env OK | torch', torch.__version__)"

In [ ]:
# 5) download YOUR model (.pth) from the private HF repo
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, shutil
tok = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(f"{RVC}/assets/weights", exist_ok=True)
shutil.copy(hf_hub_download(HF_REPO, PTH_FILE, token=tok), f"{RVC}/assets/weights/{NAME}.pth")
print("model in place")

In [ ]:
# 6) write + run the export (generator -> onnx, + download ContentVec onnx as hubert.onnx)
export_py = r"""
import os, sys, urllib.request, torch
RVC = "/kaggle/working/rvc"; OUT = "/kaggle/working/android_models"; NAME = "lecturer_ru"
os.chdir(RVC); sys.path.insert(0, RVC); os.makedirs(OUT, exist_ok=True)
_ol = torch.load
torch.load = lambda *a, **k: _ol(*a, **{**k, "weights_only": False})
# RVC export predates the dynamo exporter -> force legacy TorchScript path
_oe = torch.onnx.export
torch.onnx.export = lambda *a, **k: _oe(*a, **{**k, "dynamo": False})
from infer.modules.onnx.export import export_onnx
gen = os.path.join(OUT, "generator.onnx")
print("exporting generator ->", gen)
print(" ", export_onnx("assets/weights/%s.pth" % NAME, gen))
hub = os.path.join(OUT, "hubert.onnx")
if not os.path.exists(hub):
    url = "https://huggingface.co/NaruseMioShirakana/MoeSS-SUBModel/resolve/main/vec-768-layer-12.onnx"
    print("downloading ContentVec ->", hub); urllib.request.urlretrieve(url, hub)
import onnxruntime as ort
for nm, p in [("hubert", hub), ("generator", gen)]:
    s = ort.InferenceSession(p, providers=["CPUExecutionProvider"])
    print("\n== %s.onnx ==" % nm)
    print("  INPUTS :", [(i.name, i.shape, i.type) for i in s.get_inputs()])
    print("  OUTPUTS:", [(o.name, o.shape, o.type) for o in s.get_outputs()])
print("\nDONE")
"""
open("/kaggle/working/export.py", "w").write(export_py)
!/kaggle/working/rvc310/bin/python /kaggle/working/export.py

In [ ]:
# 7) the two files to download and push to the phone
import os
for f in ["generator.onnx", "hubert.onnx"]:
    p = os.path.join(OUT, f)
    print(f, "->", p, "| MB:", round(os.path.getsize(p)/1e6,1) if os.path.exists(p) else "MISSING")